In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras import layers
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

# Globals - update DATASET_PATH if needed
DATASET_PATH = "/content/drive/MyDrive/dataset"   # where models will be saved
os.makedirs(OUT_DIR, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

# Use mixed precision for faster training on supported GPUs
try:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision enabled.")
except Exception:
    print("Mixed precision not enabled/available.")


Mixed precision enabled.


In [ ]:
def make_datasets(folder_path, img_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED, val_split=0.20):
    # create raw datasets
    raw_train = tf.keras.preprocessing.image_dataset_from_directory(
        folder_path,
        labels='inferred',
        label_mode='int',
        validation_split=val_split,
        subset='training',
        seed=seed,
        image_size=(img_size, img_size),
        batch_size=batch_size
    )
    raw_val = tf.keras.preprocessing.image_dataset_from_directory(
        folder_path,
        labels='inferred',
        label_mode='int',
        validation_split=val_split,
        subset='validation',
        seed=seed,
        image_size=(img_size, img_size),
        batch_size=batch_size
    )

    # Extract class names BEFORE wrapping with prefetch()
    class_names = raw_train.class_names

    # Now prefetch (safe)
    train_ds = raw_train.prefetch(AUTOTUNE)
    val_ds = raw_val.prefetch(AUTOTUNE)

    return train_ds, val_ds, class_names


In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.12),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
    # don't set dtype here; final Dense will be float32
], name="data_augmentation")


In [ ]:
def build_classifier_model(num_classes, img_size=IMG_SIZE, base_name="EfficientNetV2S", dropout_rate=0.3):
    """
    Build a transfer-learning classifier using a pretrained backbone.
    base_name: "EfficientNetV2S" or "EfficientNetV2L" (swap if you have memory)
    """
    # choose backbone
    if base_name == "EfficientNetV2S":
        base = tf.keras.applications.EfficientNetV2S(include_top=False, weights="imagenet",
                                                     input_shape=(img_size, img_size, 3))
    elif base_name == "EfficientNetV2L":
        base = tf.keras.applications.EfficientNetV2L(include_top=False, weights="imagenet",
                                                     input_shape=(img_size, img_size, 3))
    else:
        raise ValueError("Unknown base_name")
    base.trainable = False  # freeze for head training

    inputs = keras.Input(shape=(img_size, img_size, 3))
    x = data_augmentation(inputs, training=True)
    x = tf.keras.applications.efficientnet_v2.preprocess_input(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout_rate)(x)
    # final dense in float32
    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)
    model = keras.Model(inputs, outputs)
    return model


In [ ]:
def compile_model(model, lr=1e-4):
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)
    opt = keras.optimizers.AdamW(learning_rate=lr, weight_decay=1e-5)
    model.compile(optimizer=opt, loss=loss_fn, metrics=["accuracy"])
    return model


In [ ]:
# Level-1 path is the root dataset with 4 top-level folders: Blossom, Fruit, Leaf, Stem
train_part_ds, val_part_ds, part_class_names = make_datasets(DATASET_PATH)
NUM_PARTS = len(part_class_names)
print("Part classes:", part_class_names)

part_model = build_classifier_model(num_classes=NUM_PARTS, img_size=IMG_SIZE, base_name="EfficientNetV2S")
part_model = compile_model(part_model, lr=1e-4)
part_model.summary()

# Train head
EPOCHS = 15
history_part = part_model.fit(
    train_part_ds,
    validation_data=val_part_ds,
    epochs=EPOCHS,
    callbacks=[keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)]
)

# Save
part_save_path = os.path.join(OUT_DIR, "part_model.keras")
part_model.save(part_save_path)
print("Saved part model to:", part_save_path)

# Evaluate and classification report
y_true = []
y_pred = []
for images, labels in val_part_ds:
    preds = part_model.predict(images)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

print(classification_report(y_true, y_pred, target_names=part_class_names))


Found 800 files belonging to 4 classes.
Using 640 files for training.
Found 800 files belonging to 4 classes.
Using 160 files for validation.
Part classes: ['Blossom', 'Fruit', 'Leaf', 'Stem']


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-s (Functional)   │ (None, 7, 7, 1280)     │    20,331,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │         5,124 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,336,484 (77.58 MB)

 Trainable params: 5,124 (20.02 KB)

 Non-trainable params: 20,331,360 (77.56 MB)

Epoch 1/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 180s 7s/step - accuracy: 0.3166 - loss: 1.3937 - val_accuracy: 0.5063 - val_loss: 1.1884
Epoch 2/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 23s 1s/step - accuracy: 0.4595 - loss: 1.2431 - val_accuracy: 0.6125 - val_loss: 1.0504
Epoch 3/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 23s 1s/step - accuracy: 0.5555 - loss: 1.1181 - val_accuracy: 0.7250 - val_loss: 0.9298
Epoch 4/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 22s 1s/step - accuracy: 0.7078 - loss: 0.9592 - val_accuracy: 0.8000 - val_loss: 0.8277
Epoch 5/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 42s 1s/step - accuracy: 0.7366 - loss: 0.8841 - val_accuracy: 0.8562 - val_loss: 0.7392
Epoch 6/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 41s 1s/step - accuracy: 0.8217 - loss: 0.7547 - val_accuracy: 0.8813 - val_loss: 0.6650
Epoch 7/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 22s 1s/step - accuracy: 0.8703 - loss: 0.6927 - val_accuracy: 0.9062 - val_loss: 0.6007
Epoch 8/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 22s 1s/step - accuracy: 0.8717 - loss: 0.6533 - val_accuracy: 0.9250 - val_loss

In [ ]:
def train_and_save_part_disease_model(part_name, base_dataset_root=DATASET_PATH, img_size=IMG_SIZE,
                                     base_name="EfficientNetV2S", epochs_head=15, epochs_finetune=12):
    """
    Train a disease classifier for a specific part (expects subfolders = disease classes).
    Saves model to DRIVE under OUT_DIR/{part_name}_model.keras
    Returns: model, class_names
    """
    folder = os.path.join(base_dataset_root, part_name)
    assert os.path.isdir(folder), f"No folder found: {folder}"

    train_ds, val_ds, class_names = make_datasets(folder)
    print(f"Training {part_name} disease model. Classes: {class_names}")

    model = build_classifier_model(num_classes=len(class_names), img_size=img_size, base_name=base_name)
    model = compile_model(model, lr=1e-4)

    # Head training
    model.fit(train_ds, validation_data=val_ds, epochs=epochs_head,
              callbacks=[keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)])

    # Fine-tune: unfreeze last 30% of backbone
    base = model.layers[3]  # note: indexing may vary; safer to search for backbone
    # safer approach: find first layer of pretrained backbone by name
    # but we'll unfreeze the whole base and then freeze first 70% by layers
    # Find backbone by type:
    backbone = None
    for l in model.layers:
        if isinstance(l, tf.keras.Model) and l.name.startswith("efficientnetv2"):
            backbone = l
            break
    if backbone is None:
        # fallback: use the first non-augmentation ConvNet-like model in the graph
        backbone = model.layers[3]

    backbone.trainable = True
    n = len(backbone.layers)
    freeze_until = int(0.7 * n)
    for i, layer in enumerate(backbone.layers):
        layer.trainable = False if i < freeze_until else True

    # recompile with lower lr
    model = compile_model(model, lr=1e-5)
    model.fit(train_ds, validation_data=val_ds, epochs=epochs_finetune,
              callbacks=[keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)])

    # Save model
    save_path = os.path.join(OUT_DIR, f"{part_name.lower().replace(' ', '_')}_model.keras")
    model.save(save_path)
    print(f"Saved {part_name} disease model to: {save_path}")

    # Eval & classification_report
    y_true = []
    y_pred = []
    for images, labels in val_ds:
        preds = model.predict(images)
        y_pred.extend(np.argmax(preds, axis=1))
        y_true.extend(labels.numpy())
    print("Evaluation for", part_name)
    print(classification_report(y_true, y_pred, target_names=class_names))

    return model, class_names, save_path


In [ ]:
blossom_model, blossom_classes, blossom_path = train_and_save_part_disease_model("Blossom")


Found 120 files belonging to 3 classes.
Using 96 files for training.
Found 120 files belonging to 3 classes.
Using 24 files for validation.
Training Blossom disease model. Classes: ['Blossom - Healthy', 'Blossom- Anthracnose', 'Blossom- Powdery Mildew']
Epoch 1/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 38s 7s/step - accuracy: 0.4010 - loss: 1.1676 - val_accuracy: 0.4583 - val_loss: 1.0514
Epoch 2/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - accuracy: 0.3776 - loss: 1.1567 - val_accuracy: 0.4583 - val_loss: 1.0439
Epoch 3/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 902ms/step - accuracy: 0.3750 - loss: 1.1123 - val_accuracy: 0.5000 - val_loss: 1.0371
Epoch 4/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 870ms/step - accuracy: 0.4440 - loss: 1.0812 - val_accuracy: 0.5000 - val_loss: 1.0302
Epoch 5/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 902ms/step - accuracy: 0.4076 - loss: 1.1384 - val_accuracy: 0.4583 - val_loss: 1.0237
Epoch 6/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - accuracy: 0.3880 - loss: 1.1089 - val_accuracy: 0.4583 - val_loss: 1.017

In [ ]:
fruit_model, fruit_classes, fruit_path = train_and_save_part_disease_model("Fruit")


Found 200 files belonging to 5 classes.
Using 160 files for training.
Found 200 files belonging to 5 classes.
Using 40 files for validation.
Training Fruit disease model. Classes: ['Fruit- Alternaria', 'Fruit- Anthracnose', 'Fruit- Black mold rot', 'Fruit- stem end rot', 'Fruit-healthy']
Epoch 1/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 40s 4s/step - accuracy: 0.1813 - loss: 1.6515 - val_accuracy: 0.3500 - val_loss: 1.6615
Epoch 2/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 436ms/step - accuracy: 0.2092 - loss: 1.6824 - val_accuracy: 0.3750 - val_loss: 1.6349
Epoch 3/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 454ms/step - accuracy: 0.2686 - loss: 1.5947 - val_accuracy: 0.3750 - val_loss: 1.6084
Epoch 4/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 454ms/step - accuracy: 0.2704 - loss: 1.6254 - val_accuracy: 0.3750 - val_loss: 1.5836
Epoch 5/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 484ms/step - accuracy: 0.3022 - loss: 1.5979 - val_accuracy: 0.3750 - val_loss: 1.5602
Epoch 6/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 678ms/step - accuracy: 0.3484 - loss: 1.5453

In [ ]:
leaf_model, leaf_classes, leaf_path = train_and_save_part_disease_model("Leaf", epochs_head=18, epochs_finetune=12)


Found 320 files belonging to 8 classes.
Using 256 files for training.
Found 320 files belonging to 8 classes.
Using 64 files for validation.
Training Leaf disease model. Classes: ['Leaf - Bacterial Canker', 'Leaf -Anthracnose', 'Leaf- Die back', 'Leaf- Gall midge', 'Leaf- Powdery Mildew', 'Leaf- Sooty mold', 'Leaf- healthy', 'Leaf-Red rust']
Epoch 1/18
8/8 ━━━━━━━━━━━━━━━━━━━━ 40s 2s/step - accuracy: 0.0962 - loss: 2.2092 - val_accuracy: 0.2344 - val_loss: 1.9960
Epoch 2/18
8/8 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.1211 - loss: 2.0898 - val_accuracy: 0.2812 - val_loss: 1.9384
Epoch 3/18
8/8 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.1375 - loss: 2.0712 - val_accuracy: 0.2969 - val_loss: 1.8813
Epoch 4/18
8/8 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.2274 - loss: 2.0013 - val_accuracy: 0.3438 - val_loss: 1.8261
Epoch 5/18
8/8 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.2838 - loss: 1.9247 - val_accuracy: 0.4062 - val_loss: 1.7715
Epoch 6/18
8/8 ━━━━━━━━━━━━━━━━━━━━ 

1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 219ms/step
Evaluation for Leaf
                         precision    recall  f1-score   support

Leaf - Bacterial Canker       0.83      1.00      0.91         5
      Leaf -Anthracnose       1.00      0.78      0.88         9
         Leaf- Die back       1.00      1.00      1.00         8
       Leaf- Gall midge       1.00      0.80      0.89         5
   Leaf- Powdery Mildew       0.77      1.00      0.87        10
       Leaf- Sooty mold       1.00      0.62      0.77         8
          Leaf- healthy       0.92      1.00      0.96        12
          Leaf-Red rust       0.88      1.00      0.93         7

               accuracy                           0.91        64
              macro avg       0.93      0.90      0.90        64
           weighted avg       0.92      0.91      0.90        64



In [ ]:
stem_model, stem_classes, stem_path = train_and_save_part_disease_model("Stem")


Found 160 files belonging to 4 classes.
Using 128 files for training.
Found 160 files belonging to 4 classes.
Using 32 files for validation.
Training Stem disease model. Classes: ['Stem- Gummosis', 'Stem- Pink disease', 'Stem-Black banded', 'Stem-healthy']
Epoch 1/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 39s 5s/step - accuracy: 0.1708 - loss: 1.5531 - val_accuracy: 0.0625 - val_loss: 1.5308
Epoch 2/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 9s 3s/step - accuracy: 0.1865 - loss: 1.5708 - val_accuracy: 0.0625 - val_loss: 1.5012
Epoch 3/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 9s 2s/step - accuracy: 0.2260 - loss: 1.5084 - val_accuracy: 0.0625 - val_loss: 1.4725
Epoch 4/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 11s 3s/step - accuracy: 0.2500 - loss: 1.4896 - val_accuracy: 0.0938 - val_loss: 1.4453
Epoch 5/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 10s 3s/step - accuracy: 0.2156 - loss: 1.4674 - val_accuracy: 0.1562 - val_loss: 1.4199
Epoch 6/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 8s 2s/step - accuracy: 0.2667 - loss: 1.4379 - val_accuracy: 0.1875 - val_loss: 1.3963
Ep

1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
Evaluation for Stem
                    precision    recall  f1-score   support

    Stem- Gummosis       1.00      0.80      0.89         5
Stem- Pink disease       1.00      0.89      0.94         9
 Stem-Black banded       0.80      0.89      0.84         9
      Stem-healthy       0.80      0.89      0.84         9

          accuracy                           0.88        32
         macro avg       0.90      0.87      0.88        32
      weighted avg       0.89      0.88      0.88        32



In [ ]:
# load saved models (if not in memory) and prepare mappings
from pathlib import Path

# If you have the models in variables already (from training), you can reuse them.
# Otherwise load from disk:
part_model = keras.models.load_model(os.path.join(OUT_DIR, "part_model.keras"))

blossom_model = keras.models.load_model(blossom_path)
fruit_model   = keras.models.load_model(fruit_path)
leaf_model    = keras.models.load_model(leaf_path)
stem_model    = keras.models.load_model(stem_path)

part_name_map = part_class_names  # e.g. ['Blossom','Fruit','Leaf','Stem']
part_to_model = {
    "Blossom": (blossom_model, blossom_classes),
    "Fruit":   (fruit_model, fruit_classes),
    "Leaf":    (leaf_model, leaf_classes),
    "Stem":    (stem_model, stem_classes)
}

def hierarchical_predict(image_path):
    # load & preprocess
    img = keras.preprocessing.image.load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE))
    arr = keras.preprocessing.image.img_to_array(img)
    arr = np.expand_dims(arr, 0)
    arr_pre = tf.keras.applications.efficientnet_v2.preprocess_input(arr)

    # Level 1
    part_preds = part_model.predict(arr_pre)
    part_idx = int(np.argmax(part_preds, axis=1)[0])
    part_name = part_name_map[part_idx]
    part_conf = float(np.max(part_preds))

    # Level 2
    model2, class_list = part_to_model[part_name]
    disease_preds = model2.predict(arr_pre)
    disease_idx = int(np.argmax(disease_preds, axis=1)[0])
    disease_name = class_list[disease_idx]
    disease_conf = float(np.max(disease_preds))

    return {
        "part": part_name,
        "part_confidence": part_conf,
        "disease": disease_name,
        "disease_confidence": disease_conf
    }

# Example:
# print(hierarchical_predict("/content/some_leaf_image.jpg"))


NameError: name 'keras' is not defined

In [ ]:
# Quick tips:
# - If validation is noisy, increase augmentation or use KFold cross-validation.
# - For best performance you can swap base_name="EfficientNetV2L" inside build_classifier_model
#   but ensure you have enough GPU memory.
# - To export TFLite for mobile: use model.export(...) on Keras 3 or tf.saved_model save then TFLiteConverter.

# Save a small metadata JSON with class lists and paths for easy deployment
import json
meta = {
    "part_classes": part_class_names,
    "blossom_classes": blossom_classes,
    "fruit_classes": fruit_classes,
    "leaf_classes": leaf_classes,
    "stem_classes": stem_classes,
    "models": {
        "part": os.path.join(OUT_DIR, "part_model.keras"),
        "blossom": blossom_path,
        "fruit": fruit_path,
        "leaf": leaf_path,
        "stem": stem_path
    }
}
with open(os.path.join(OUT_DIR, "hierarchy_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print("Metadata saved to:", os.path.join(OUT_DIR, "hierarchy_meta.json"))
